In [18]:

from dotenv import load_dotenv, find_dotenv
import os

from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_openai import ChatOpenAI



from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnablePassthrough
from langchain_core.caches import InMemoryCache
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import (
    AIMessage, 
    HumanMessage, 
    SystemMessage
)
from langchain_core.prompts import (
    ChatPromptTemplate,
    PromptTemplate,
    SystemMessagePromptTemplate,
    AIMessagePromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder
)

import langchain
from pydantic import BaseModel, Field
from typing import List
from operator import itemgetter

from langchain_community.chat_message_histories.in_memory import ChatMessageHistory
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import *
from langchain_text_splitters import CharacterTextSplitter


from langchain.tools import tool
from langchain.agents import create_agent

from langchain_pinecone import PineconeVectorStore

In [2]:
load_dotenv(find_dotenv())
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large", openai_api_key=os.environ.get("OPENAI_API_KEY"))

In [4]:
llm = ChatOpenAI(model="gpt-5-nano-2025-08-07", openai_api_key=OPENAI_API_KEY)

In [6]:
vectorstore = PineconeVectorStore(
    index_name=os.environ["INDEX_NAME"], embedding=embeddings
)

In [7]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [8]:

prompt_template = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:

{context}

Question: {question}

Provide a detailed answer:"""
)

In [10]:
def format_docs(docs):
    """Format retrieved documents into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)

In [9]:
# ============================================================================
# IMPLEMENTATION 1: Without LCEL (Simple Function-Based Approach)
# ============================================================================
def retrieval_chain_without_lcel(query: str):
    """
    Simple retrieval chain without LCEL.
    Manually retrieves documents, formats them, and generates a response.

    Limitations:
    - Manual step-by-step execution
    - No built-in streaming support
    - No async support without additional code
    - Harder to compose with other chains
    - More verbose and error-prone
    """
    # Step 1: Retrieve relevant documents
    docs = retriever.invoke(query)

    # Step 2: Format documents into context string
    context = format_docs(docs)

    # Step 3: Format the prompt with context and question
    messages = prompt_template.format_messages(context=context, question=query)

    # Step 4: Invoke LLM with the formatted messages
    response = llm.invoke(messages)

    # Step 5: Return the content
    return response.content



In [19]:

# ============================================================================
# IMPLEMENTATION 2: With LCEL (LangChain Expression Language) - BETTER APPROACH
# ============================================================================
def create_retrieval_chain_with_lcel():
    """
    Create a retrieval chain using LCEL (LangChain Expression Language).
    Returns a chain that can be invoked with {"question": "..."}

    Advantages over non-LCEL approach:
    - Declarative and composable: Easy to chain operations with pipe operator (|)
    - Built-in streaming: chain.stream() works out of the box
    - Built-in async: chain.ainvoke() and chain.astream() available
    - Batch processing: chain.batch() for multiple inputs
    - Type safety: Better integration with LangChain's type system
    - Less code: More concise and readable
    - Reusable: Chain can be saved, shared, and composed with other chains
    - Better debugging: LangChain provides better observability tools
    """
    retrieval_chain = (
        RunnablePassthrough.assign(
            context=itemgetter("question") | retriever | format_docs
        )
        | prompt_template
        | llm
        | StrOutputParser()
    )
    return retrieval_chain

In [12]:
# Query
query = "what is Pinecone in machine learning?"

In [13]:
result_without_lcel = retrieval_chain_without_lcel(query)
print(result_without_lcel)

Pinecone is a fully managed, cloud-based vector database tailored for machine learning workflows. It stores vector embeddings and provides fast, scalable retrieval of similar data points, making it easy to build large-scale ML applications that rely on vector similarity search.

Key points about Pinecone in ML:
- Fast and scalable: designed to handle large data sets with millions or billions of points.
- Efficient retrieval: enables quick search for similar vectors, supporting high query throughput and low latency.
- Managed infrastructure: Pinecone takes care of the underlying infrastructure and maintenance, so you can focus on model and application development.
- Secure: meets the security needs of businesses and organizations.
- Easy to use: offers a simple API for storing and retrieving vector data, facilitating integration into existing ML workflows.
- Real-time updates: supports live updates to the vector database as new data points are added, keeping data current.
- Data integra

In [ ]:
chain_with_lcel = create_retrieval_chain_with_lcel()
result_with_lcel = chain_with_lcel.invoke({"question": query})
print("\nAnswer:")
print(result_with_lcel)


Answer:
Pinecone is a fully managed cloud-based vector database designed for machine learning workflows. It focuses on fast, scalable retrieval of items based on vector representations, making it easy to power large-scale similarity search tasks in ML applications.

Key points about Pinecone in ML:
- Purpose and capabilities
  - Stores and retrieves vector embeddings and associated metadata.
  - Enables efficient retrieval of similar data points using their vector representations.
  - Designed to support large-scale ML applications with millions or billions of data points.

- Performance and scale
  - Optimized for high query throughput and low latency search, even at large scales.
  - Handles real-time updates, so the vector database stays up-to-date as new data points are added.

- Operational characteristics
  - Fully managed infrastructure, reducing maintenance burden for users.
  - Secure platform that meets common security requirements for businesses and organizations.

- Integr

: 